# 🌐 Interactive Mapping with Folium — Exercise Notebook

---

> **How to use this notebook:**  
> Each section contains **worked examples** you can run, followed by **practice exercises**.  
> Read the examples carefully, then attempt the exercises in the solution cells.  
> Use `Shift + Enter` to run a cell.

---

## 📋 Table of Contents

| # | Topic |
|---|-------|
| 0 | Setup & Environment |
| 1 | Basic Map Initialization |
| 2 | Markers & Popups |
| 3 | Customizing Markers with Icons |
| 4 | Routing with OpenRouteService API |
| 5 | Parsing GeoJSON & Plotting Routes |
| 6 | Saving Maps as HTML |
| 7 | Downloading Data with pygris |
| 8 | Multi-Layer Maps with `.explore()` |
| 9 | Road Network Categorization & Styling |
| 10 | Layer Controls & Basemap Switching |
| 11 | Bonus — Circles, Polygons & Heatmaps |

---

# 0️⃣ Setup & Environment

---

> This notebook runs on **Google Colab**.  
> All maps are rendered as interactive **Leaflet.js** HTML inside the notebook and can be saved as standalone `.html` files.

| Library | Purpose |
|---------|----------|
| `folium` | Create interactive Leaflet.js maps in Python |
| `geopandas` | Read and manipulate vector geospatial data |
| `pandas` | Tabular data manipulation |
| `requests` | HTTP calls (e.g. routing API) |
| `pygris` | Download US Census TIGER shapefiles |
| `mapclassify` | Classification schemes for choropleth maps |

---

### 💡 Example 0.1 — Mount Drive & Create Folders

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DataFolder   = 'data'
OutputFolder = 'output'

for folder in [DataFolder, OutputFolder]:
    if not os.path.exists(folder):
        os.mkdir(folder)
        print(f'Created: {folder}')
    else:
        print(f'Already exists: {folder}')

### 💡 Example 0.2 — Import All Libraries

In [ ]:
# !pip install folium geopandas pygris mapclassify   # uncomment if needed

import pandas as pd
import geopandas as gpd
import folium
import json
import requests
import os
from folium import Figure

print('Folium version  :', folium.__version__)
print('GeoPandas version:', gpd.__version__)
print('All libraries loaded ✅')

# 1️⃣ Basic Map Initialization

---

> `folium.Map()` creates a Leaflet map centred on a coordinate with a starting zoom level.  
> Wrap it in a `Figure` object to control the rendered size inside a notebook.

| Parameter | Description |
|-----------|-------------|
| `location=[lat, lon]` | Map centre coordinate |
| `zoom_start` | Initial zoom (1=world, 18=street) |
| `tiles` | Basemap tile provider |
| `width` / `height` | Map size in pixels |

**Common tile providers:**
| Tile name | Style |
|-----------|-------|
| `'OpenStreetMap'` | Standard street map |
| `'CartoDB positron'` | Clean light grey |
| `'CartoDB dark_matter'` | Dark background |
| `'Stamen Terrain'` | Terrain / elevation |
| `'Esri.WorldImagery'` (via plugin) | Satellite |

---

### 💡 Example 1.1 — Create a Basic Folium Map

In [ ]:
import folium
from folium import Figure

# Wrap in a Figure for controlled sizing in the notebook
fig = Figure(width=800, height=400)

# Create a map centred on the continental US
m = folium.Map(location=[39.83, -98.58], zoom_start=4)
fig.add_child(m)

# Display
fig

### 💡 Example 1.2 — Different Basemaps

In [ ]:
import folium
from folium import Figure

basemaps = [
    ('OpenStreetMap',      'Standard Street Map'),
    ('CartoDB positron',   'Clean Light Grey'),
    ('CartoDB dark_matter','Dark Background'),
]

# Show the first three basemaps in sequence
for tiles, label in basemaps:
    print(f'--- {label} ({tiles}) ---')
    fig = Figure(width=500, height=300)
    m   = folium.Map(location=[46.88, -96.79], zoom_start=8, tiles=tiles)
    fig.add_child(m)
    display(fig)

### 💡 Example 1.3 — Map Centred on North Dakota with Zoom

In [ ]:
import folium
from folium import Figure

fig = Figure(width=800, height=500)
m   = folium.Map(
    location=[47.5, -100.5],   # approx. centre of North Dakota
    zoom_start=7,
    tiles='CartoDB positron'
)
fig.add_child(m)
fig

# 2️⃣ Markers & Popups

---

> **Markers** pin a location on the map.  
> **Popups** show rich HTML content when clicked.  
> **Tooltips** show text on hover.

| Element | Code |
|---------|------|
| Basic marker | `folium.Marker([lat, lon]).add_to(m)` |
| With popup | `folium.Marker(..., popup='Text')` |
| With tooltip | `folium.Marker(..., tooltip='Hover text')` |
| HTML popup | `folium.Popup('<b>Bold</b>', max_width=200)` |
| Circle marker | `folium.CircleMarker([lat,lon], radius=8, color='red')` |
| Circle | `folium.Circle([lat,lon], radius=5000, color='blue')` |

---

### 💡 Example 2.1 — Basic Markers with Popup & Tooltip

In [ ]:
import folium
from folium import Figure

FargoCoords   = (46.8772, -96.7898)
NewYorkCoords = (40.6610, -73.9440)

fig = Figure(width=800, height=400)
m   = folium.Map(location=[39.83, -98.58], zoom_start=4)

# Marker with popup (click) and tooltip (hover)
folium.Marker(
    FargoCoords,
    popup=folium.Popup('<b>Fargo, ND</b><br>Pop: ~125,000', max_width=200),
    tooltip='Click for info'
).add_to(m)

folium.Marker(
    NewYorkCoords,
    popup=folium.Popup('<b>New York, NY</b><br>Pop: ~8,300,000', max_width=200),
    tooltip='Click for info'
).add_to(m)

fig.add_child(m)
fig

### 💡 Example 2.2 — Circle & CircleMarker

In [ ]:
import folium
from folium import Figure

fig = Figure(width=800, height=450)
m   = folium.Map(location=[46.88, -96.79], zoom_start=9)

# Circle: radius in metres — useful for spatial buffers
folium.Circle(
    location=FargoCoords,
    radius=20_000,           # 20 km
    color='blue',
    fill=True,
    fill_opacity=0.15,
    tooltip='20 km radius around Fargo'
).add_to(m)

# CircleMarker: radius in pixels — size stays constant regardless of zoom
folium.CircleMarker(
    location=FargoCoords,
    radius=10,
    color='red',
    fill=True,
    fill_color='red',
    popup='Fargo city centre'
).add_to(m)

fig.add_child(m)
fig

### 💡 Example 2.3 — Multiple Markers from a List

In [ ]:
import folium
from folium import Figure

nd_cities = [
    ('Fargo',       46.8772, -96.7898, 125_000),
    ('Bismarck',    46.8083, -100.7837, 74_000),
    ('Grand Forks', 47.9253, -97.0329,  59_000),
    ('Minot',       48.2325, -101.2963, 48_000),
    ('Williston',   48.1470, -103.6180, 29_000),
]

fig = Figure(width=800, height=500)
m   = folium.Map(location=[47.5, -100.5], zoom_start=7, tiles='CartoDB positron')

for city, lat, lon, pop in nd_cities:
    folium.Marker(
        [lat, lon],
        popup=folium.Popup(f'<b>{city}</b><br>Population: {pop:,}', max_width=150),
        tooltip=city
    ).add_to(m)

fig.add_child(m)
fig

# 3️⃣ Customizing Markers with Icons

---

> `folium.Icon` lets you change marker colour and use icons from **Bootstrap** or **FontAwesome**.

| Parameter | Options |
|-----------|----------|
| `color` | `'red'`, `'blue'`, `'green'`, `'purple'`, `'orange'`, `'gray'`, `'black'`, `'white'`, `'beige'`, `'darkblue'`, `'darkred'`, `'lightred'`, `'darkgreen'`, `'lightgreen'`, `'cadetblue'` |
| `icon` | Any Bootstrap glyph or FA icon name (e.g. `'home'`, `'star'`, `'car'`) |
| `prefix` | `'glyphicon'` (Bootstrap, default) or `'fa'` (FontAwesome) |

> 🔗 Browse FontAwesome icons: [fontawesome.com/icons](https://fontawesome.com/v4/icons/)  
> 🔗 Browse Bootstrap icons: [getbootstrap.com/docs/3.4/components/#glyphicons](https://getbootstrap.com/docs/3.4/components/#glyphicons)

---

### 💡 Example 3.1 — Custom Icon Colours

In [ ]:
import folium
from folium import Figure

FargoCoords   = (46.8772, -96.7898)
NewYorkCoords = (40.6610, -73.9440)

fig = Figure(width=800, height=400)
m   = folium.Map(location=[39.83, -98.58], zoom_start=4)

# Green crosshairs for Fargo
folium.Marker(
    FargoCoords,
    popup='Fargo, ND',
    icon=folium.Icon(color='green', icon='crosshairs', prefix='fa')
).add_to(m)

# Red crosshairs for New York
folium.Marker(
    NewYorkCoords,
    popup='New York, NY',
    icon=folium.Icon(color='red', icon='crosshairs', prefix='fa')
).add_to(m)

fig.add_child(m)
fig

### 💡 Example 3.2 — Themed Icons for Different Feature Types

In [ ]:
import folium
from folium import Figure

features = [
    ('Fargo Airport',    46.9207, -96.8157, 'plane',    'darkblue',  'fa'),
    ('Fargo Hospital',   46.8805, -96.8006, 'hospital', 'red',       'fa'),
    ('Fargo City Hall',  46.8757, -96.7897, 'building', 'orange',    'fa'),
    ('NDSU Campus',      46.8969, -96.8019, 'graduation-cap','purple','fa'),
]

fig = Figure(width=800, height=450)
m   = folium.Map(location=[46.88, -96.80], zoom_start=12, tiles='CartoDB positron')

for name, lat, lon, icon, color, prefix in features:
    folium.Marker(
        [lat, lon],
        popup=name,
        tooltip=name,
        icon=folium.Icon(color=color, icon=icon, prefix=prefix)
    ).add_to(m)

fig.add_child(m)
fig

### 💡 Example 3.3 — DivIcon for Custom HTML Markers

In [ ]:
import folium
from folium import Figure
from folium.features import DivIcon

fig = Figure(width=800, height=450)
m   = folium.Map(location=[47.5, -100.5], zoom_start=7, tiles='CartoDB positron')

nd_cities = [
    ('Fargo',       46.8772, -96.7898, 125_000),
    ('Bismarck',    46.8083, -100.7837, 74_000),
    ('Grand Forks', 47.9253, -97.0329,  59_000),
]

for city, lat, lon, pop in nd_cities:
    # Standard marker
    folium.CircleMarker([lat, lon], radius=8, color='navy', fill=True,
                        fill_color='steelblue', fill_opacity=0.8).add_to(m)
    # Custom HTML label via DivIcon
    folium.Marker(
        [lat, lon],
        icon=DivIcon(
            icon_size=(120, 36),
            icon_anchor=(0, 0),
            html=f'<div style="font-size:11px;font-weight:bold;'
                 f'color:navy;white-space:nowrap">{city}<br><span '
                 f'style="font-weight:normal;color:grey">{pop:,}</span></div>'
        )
    ).add_to(m)

fig.add_child(m)
fig

# 4️⃣ Routing with OpenRouteService (ORS) API

---

> **OpenRouteService** provides free routing, isochrones, and geocoding via a REST API.  
> You need a free API key from [account.heigit.org/signup](https://account.heigit.org/signup).

| Step | Description |
|------|-------------|
| 1 | Sign up and copy your **Basic Key** |
| 2 | Build the request `params` dict |
| 3 | Call `requests.get(url, params=params)` |
| 4 | Check `response.status_code == 200` |
| 5 | Parse the returned **GeoJSON** |

> ⚠️ **Important:** ORS expects coordinates as **(Longitude, Latitude)** — the reverse of Folium's **(Latitude, Longitude)**.

---

### 💡 Example 4.1 — Query the ORS Directions API

In [ ]:
import requests

# Replace with your own key from account.heigit.org
ORS_API_KEY = 'YOUR_API_KEY_HERE'

FargoCoords   = (46.8772, -96.7898)
NewYorkCoords = (40.6610, -73.9440)

# ORS expects lon,lat (reversed!)
Parameters = {
    'api_key': ORS_API_KEY,
    'start'  : f'{FargoCoords[1]},{FargoCoords[0]}',
    'end'    : f'{NewYorkCoords[1]},{NewYorkCoords[0]}'
}

Response = requests.get(
    'https://api.openrouteservice.org/v2/directions/driving-car',
    params=Parameters
)

print(f'Status code : {Response.status_code}')
if Response.status_code == 200:
    RouteData = Response.text
    print('Route data received ✅')
else:
    RouteData = None
    print('Request failed ❌ — check your API key')

### 💡 Example 4.2 — Different Travel Profiles

In [ ]:
# ORS supports multiple routing profiles:
# 'driving-car'  → fastest car route
# 'driving-hgv'  → heavy goods vehicle
# 'cycling-road' → road cycling
# 'foot-walking' → pedestrian

# To use a different profile, change the URL segment:
# 'https://api.openrouteservice.org/v2/directions/cycling-road'

# Example (requires valid API key):
# profile   = 'cycling-road'
# base_url  = f'https://api.openrouteservice.org/v2/directions/{profile}'
# response  = requests.get(base_url, params=Parameters)

print('Available ORS profiles:')
for p in ['driving-car','driving-hgv','cycling-road','cycling-mountain',
           'cycling-electric','foot-walking','foot-hiking','wheelchair']:
    print(f'  • {p}')

# 5️⃣ Parsing GeoJSON & Plotting Routes

---

> The ORS API returns a **GeoJSON FeatureCollection**.  
> Parse it with `json.loads()`, extract the summary, and display the route on the map with `folium.GeoJson()`.

| Step | Code |
|------|------|
| Parse JSON | `parsed = json.loads(route_text)` |
| Get summary | `parsed['features'][0]['properties']['summary']` |
| Distance (km) | `summary['distance'] / 1000` |
| Duration (hrs) | `summary['duration'] / 3600` |
| Plot route | `folium.GeoJson(route_text, tooltip=...).add_to(m)` |

---

### 💡 Example 5.1 — Parse ORS Response & Extract Route Info

In [ ]:
import json

# Assumes RouteData was obtained in Section 4
if RouteData:
    ParsedData = json.loads(RouteData)

    # Navigate the GeoJSON structure
    Summary = ParsedData['features'][0]['properties']['summary']

    DistanceKm  = round(Summary['distance'] / 1000, 1)
    DurationHrs = round(Summary['duration'] / 3600, 1)

    print(f'Distance : {DistanceKm} km')
    print(f'Duration : {DurationHrs} hours')
    print(f'\nFull summary dict:\n{Summary}')
else:
    print('No route data — run Section 4 first with a valid API key.')

### 💡 Example 5.2 — Plot Route on a Folium Map

In [ ]:
import folium, json
from folium import Figure

FargoCoords   = (46.8772, -96.7898)
NewYorkCoords = (40.6610, -73.9440)

fig = Figure(width=800, height=450)
m   = folium.Map(location=[39.83, -98.58], zoom_start=4)

# Add origin and destination markers
for coord, label, color in [
    (FargoCoords,   'Fargo, ND',    'green'),
    (NewYorkCoords, 'New York, NY', 'red')
]:
    folium.Marker(
        coord, popup=label,
        icon=folium.Icon(color=color, icon='crosshairs', prefix='fa')
    ).add_to(m)

# Plot the GeoJSON route
if RouteData:
    ParsedData  = json.loads(RouteData)
    Summary     = ParsedData['features'][0]['properties']['summary']
    DistanceKm  = round(Summary['distance'] / 1000)
    DurationHrs = round(Summary['duration'] / 3600, 1)
    RouteTooltip = f'🚗 {DistanceKm} km | ⏱ {DurationHrs} hrs'

    folium.GeoJson(
        RouteData,
        tooltip=RouteTooltip,
        style_function=lambda x: {
            'color': '#1f78b4',
            'weight': 4,
            'opacity': 0.8
        },
        smooth_factor=1
    ).add_to(m)

fig.add_child(m)
fig

### 💡 Example 5.3 — Compare Two Routes on the Same Map

In [ ]:
# Conceptual: compare driving-car vs cycling-road routes
# (both require valid ORS API key)

import folium, json
from folium import Figure

FargoCoords   = (46.8772, -96.7898)
NewYorkCoords = (40.6610, -73.9440)

fig = Figure(width=800, height=450)
m   = folium.Map(location=[39.83, -98.58], zoom_start=4)

route_styles = [
    (RouteData, '#1f78b4', 4, 'Driving Route'),   # blue, thick
    # (CyclingRouteData, '#33a02c', 3, 'Cycling Route'),  # green
]

for route, color, weight, name in route_styles:
    if route:
        folium.GeoJson(
            route,
            name=name,
            style_function=lambda x, c=color, w=weight: {
                'color': c, 'weight': w, 'opacity': 0.8
            }
        ).add_to(m)

folium.LayerControl().add_to(m)
fig.add_child(m)
fig

# 6️⃣ Saving Maps as HTML

---

> Folium maps are **standalone HTML files** — share them with anyone, no Python required.

| Method | Code |
|--------|------|
| Save map | `m.save('output/mymap.html')` |
| Save figure | `fig.save('output/mymap.html')` |
| Check file size | `os.path.getsize(path)` |
| Download in Colab | `from google.colab import files; files.download(path)` |

---

### 💡 Example 6.1 — Save a Map & Download in Colab

In [ ]:
import folium, os
from folium import Figure

fig = Figure(width=800, height=400)
m   = folium.Map(location=[39.83, -98.58], zoom_start=4)

folium.Marker([46.8772, -96.7898], popup='Fargo').add_to(m)
folium.Marker([40.6610, -73.9440], popup='New York').add_to(m)

fig.add_child(m)

# Save to disk
OutputPath = os.path.join('output', 'directions.html')
m.save(OutputPath)

print(f'Saved to: {OutputPath}')
print(f'File size: {os.path.getsize(OutputPath)//1024} KB')

# Download to your local machine (Colab only)
# from google.colab import files
# files.download(OutputPath)

# 7️⃣ Downloading Data with pygris

---

> **pygris** downloads US Census TIGER shapefiles directly into GeoDataFrames — no manual download needed.

| Function | Data returned |
|----------|---------------|
| `pygris.counties(state='ND')` | County boundaries |
| `pygris.tracts(state='ND')` | Census tracts |
| `pygris.block_groups(state='ND', county='017')` | Block groups |
| `pygris.transportation.primary_secondary_roads(state='38')` | Roads |
| `pygris.places(state='ND')` | Populated places |
| `pygris.water.area_water(state='38', county='017')` | Water bodies |

---

### 💡 Example 7.1 — Download ND Counties & Roads

In [ ]:
# !pip install pygris   # uncomment if needed
import pygris

# Download county boundaries for North Dakota
NorthDakotaCounties = pygris.counties(state='ND')
print(f'Counties downloaded  : {len(NorthDakotaCounties)}')
print(f'CRS                  : {NorthDakotaCounties.crs}')
print(f'Columns              : {NorthDakotaCounties.columns.tolist()}')

# Download primary/secondary roads for ND
NorthDakotaRoads = pygris.transportation.primary_secondary_roads(state='38', year=2024)
print(f'\nRoad features        : {len(NorthDakotaRoads)}')
print(f'Road columns         : {NorthDakotaRoads.columns.tolist()}')

### 💡 Example 7.2 — Download Census Tracts & Block Groups

In [ ]:
import pygris

# Census tracts for all of ND
nd_tracts = pygris.tracts(state='ND')
print(f'Tracts: {len(nd_tracts)}')

# Block groups for Cass County only
cass_bg = pygris.block_groups(state='ND', county='017')
print(f'Cass County block groups: {len(cass_bg)}')

# Quick preview
cass_bg.head(3)

# 8️⃣ Multi-Layer Maps with `.explore()`

---

> `GeoDataFrame.explore()` is a one-liner that creates a rich Folium map from a GeoDataFrame.  
> Pass `m=existing_map` to stack multiple layers on the same map.

| Parameter | Description |
|-----------|-------------|
| `m=` | Existing Folium map to add layer to |
| `color=` | Uniform colour for the layer |
| `column=` | Attribute column to colour by |
| `cmap=` | Colormap |
| `categorical=True` | Treat column as categories |
| `tooltip=` | List of columns shown on hover |
| `tooltip_kwds=` | `{'aliases': ['Label1', 'Label2']}` |
| `style_kwds=` | `{'fillOpacity': 0.3, 'weight': 1}` |
| `name=` | Layer name in LayerControl |

---

### 💡 Example 8.1 — Initialize Map & Fit to Data Bounds

In [ ]:
import folium
import pygris
from folium import Figure

# !pip install mapclassify   # needed for column-based styling

NorthDakotaCounties = pygris.counties(state='ND')

# Get bounding box [minx, miny, maxx, maxy]
Bounds = NorthDakotaCounties.total_bounds

fig = Figure(width=800, height=450)
m   = folium.Map(tiles='CartoDB positron')

# Fit map view to data extent
# folium expects [[lat_min, lon_min], [lat_max, lon_max]]
m.fit_bounds([[Bounds[1], Bounds[0]], [Bounds[3], Bounds[2]]])

# Add county layer
NorthDakotaCounties.explore(
    m=m,
    color='black',
    style_kwds={'fillOpacity': 0.2, 'weight': 0.7},
    tooltip=['NAME', 'COUNTYFP'],
    tooltip_kwds={'aliases': ['County', 'FIPS']},
    name='ND Counties'
)

fig.add_child(m)
fig

### 💡 Example 8.2 — Choropleth with `.explore(column=...)`

In [ ]:
import folium, pygris
from folium import Figure

NorthDakotaCounties = pygris.counties(state='ND')
NorthDakotaCounties = NorthDakotaCounties.to_crs(5070)
NorthDakotaCounties['AreaKm2'] = NorthDakotaCounties.geometry.area / 1e6
NorthDakotaCounties = NorthDakotaCounties.to_crs(4326)  # back to WGS84 for Folium

Bounds = NorthDakotaCounties.total_bounds
fig    = Figure(width=800, height=450)
m      = folium.Map(tiles='CartoDB positron')
m.fit_bounds([[Bounds[1], Bounds[0]], [Bounds[3], Bounds[2]]])

NorthDakotaCounties.explore(
    m=m,
    column='AreaKm2',
    cmap='YlOrRd',
    tooltip=['NAME', 'AreaKm2'],
    tooltip_kwds={'aliases': ['County', 'Area (km²)']},
    legend=True,
    name='County Area'
)

fig.add_child(m)
fig

# 9️⃣ Road Network Categorization & Styling

---

> Road features have an `RTTYP` (Route Type) code.  
> We create a new `Category` column to label roads as **Interstate**, **State**, or **Other**,  
> then use this to colour the road layer on the map.

| RTTYP code | Meaning |
|------------|---------|
| `'I'` | Interstate highway |
| `'S'` | State route |
| `'U'` | US route |
| `'C'` | County road |
| `'M'` | Common name |

---

### 💡 Example 9.1 — Categorize Roads

In [ ]:
import pygris

NorthDakotaRoads = pygris.transportation.primary_secondary_roads(state='38', year=2024)

def GetCategory(Row):
    """Categorize road by RTTYP code."""
    ref = str(Row['RTTYP'])
    if 'I' in ref:   return 'Interstate'
    elif 'S' in ref: return 'State'
    elif 'U' in ref: return 'US Route'
    else:            return 'Other'

NorthDakotaRoads['Category'] = NorthDakotaRoads.apply(GetCategory, axis=1)

print('Road category counts:')
print(NorthDakotaRoads['Category'].value_counts())
NorthDakotaRoads.head(4)

### 💡 Example 9.2 — Plot Road Categories with `.explore()`

In [ ]:
import folium, pygris
from folium import Figure

NorthDakotaCounties = pygris.counties(state='ND')
NorthDakotaRoads    = pygris.transportation.primary_secondary_roads(state='38', year=2024)
NorthDakotaRoads['Category'] = NorthDakotaRoads.apply(GetCategory, axis=1)

Bounds = NorthDakotaCounties.total_bounds
fig    = Figure(width=800, height=450)
m      = folium.Map(tiles='CartoDB positron')
m.fit_bounds([[Bounds[1], Bounds[0]], [Bounds[3], Bounds[2]]])

# Counties layer
NorthDakotaCounties.explore(
    m=m, color='black',
    style_kwds={'fillOpacity': 0.1, 'weight': 0.6},
    name='Counties'
)

# Roads layer — coloured by category
NorthDakotaRoads.explore(
    m=m,
    column='Category',
    categories=['Interstate', 'State', 'US Route', 'Other'],
    cmap=['#1f78b4', '#e31a1c', '#ff7f00', 'gray'],
    categorical=True,
    tooltip=['RTTYP', 'FULLNAME', 'Category'],
    tooltip_kwds={'aliases': ['Code', 'Name', 'Category']},
    name='Roads by Category'
)

folium.LayerControl().add_to(m)
fig.add_child(m)
fig

# 🔟 Layer Controls & Basemap Switching

---

> `folium.LayerControl()` adds a toggle widget to the map so users can turn layers on/off.  
> Additional basemaps can be added as `TileLayer` objects.

| Element | Code |
|---------|------|
| Add layer control | `folium.LayerControl().add_to(m)` |
| Add extra basemap | `folium.TileLayer('CartoDB dark_matter', name='Dark').add_to(m)` |
| Named layer (for toggle) | pass `name='Layer Name'` to `.explore()` or `folium.GeoJson(name=...)` |
| Show control | call `folium.LayerControl()` **after** all layers are added |

---

### 💡 Example 10.1 — Full Multi-Layer Map with Layer Control

In [ ]:
import folium, pygris
from folium import Figure

counties = pygris.counties(state='ND')
roads    = pygris.transportation.primary_secondary_roads(state='38', year=2024)
roads['Category'] = roads.apply(GetCategory, axis=1)

bounds = counties.total_bounds
fig    = Figure(width=900, height=550)

# Default basemap
m = folium.Map(tiles='CartoDB positron')
m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])

# Additional basemap options (users can switch)
folium.TileLayer('OpenStreetMap',        name='OpenStreetMap').add_to(m)
folium.TileLayer('CartoDB dark_matter',  name='Dark Matter').add_to(m)

# County layer
counties.explore(
    m=m, color='black',
    style_kwds={'fillOpacity': 0.1, 'weight': 0.6},
    tooltip=['NAME'], name='Counties'
)

# Roads — coloured by category
roads.explore(
    m=m,
    column='Category',
    categories=['Interstate', 'State', 'US Route', 'Other'],
    cmap=['#1f78b4', '#e31a1c', '#ff7f00', 'gray'],
    categorical=True,
    tooltip=['FULLNAME', 'Category'],
    tooltip_kwds={'aliases': ['Road Name', 'Type']},
    name='Roads'
)

# Interstate highlight layer
roads[roads['Category'] == 'Interstate'].explore(
    m=m,
    color='#ffff00',
    style_kwds={'weight': 3.5},
    tooltip=['FULLNAME'],
    name='Interstate Highlight'
)

# Layer control — must be added last
folium.LayerControl(collapsed=False).add_to(m)

fig.add_child(m)
fig

### 💡 Example 10.2 — Save the Final Multi-Layer Map

In [ ]:
import os

output_path = 'output/nd_multilayer_map.html'
m.save(output_path)

print(f'Map saved: {output_path}')
print(f'File size: {os.path.getsize(output_path)//1024} KB')

# 1️⃣1️⃣ Bonus — Circles, Polygons, Polylines & Heatmaps

---

> Folium supports a variety of geometric overlays and plugins.  
> The `HeatMap` plugin visualizes point density as a smooth gradient.

| Feature | Code |
|---------|------|
| Circle (metres) | `folium.Circle([lat,lon], radius=5000)` |
| CircleMarker (pixels) | `folium.CircleMarker([lat,lon], radius=8)` |
| Polygon | `folium.Polygon([[lat,lon],...], color=...)` |
| Polyline | `folium.PolyLine([[lat,lon],...])` |
| Rectangle | `folium.Rectangle([[lat1,lon1],[lat2,lon2]])` |
| Heatmap | `from folium.plugins import HeatMap; HeatMap(data).add_to(m)` |
| Marker cluster | `from folium.plugins import MarkerCluster` |

---

### 💡 Example 11.1 — Shapes: Circle, Polygon, Polyline, Rectangle

In [ ]:
import folium
from folium import Figure

fig = Figure(width=800, height=500)
m   = folium.Map(location=[47.0, -100.5], zoom_start=7, tiles='CartoDB positron')

# Circle — 50 km radius around Bismarck
folium.Circle([46.8083, -100.7837], radius=50_000,
              color='blue', fill=True, fill_opacity=0.1,
              tooltip='50 km from Bismarck').add_to(m)

# Polygon — triangle
folium.Polygon(
    locations=[[48.0, -100.0], [47.0, -102.0], [46.0, -100.5]],
    color='green', fill=True, fill_opacity=0.3,
    tooltip='Custom polygon'
).add_to(m)

# Polyline — route between cities
folium.PolyLine(
    locations=[[46.8772, -96.7898], [46.8083, -100.7837], [48.2325, -101.2963]],
    color='red', weight=2, dash_array='5',
    tooltip='City route'
).add_to(m)

# Rectangle — bounding box
folium.Rectangle(
    bounds=[[46.5, -97.5], [47.5, -96.0]],
    color='purple', fill=False,
    tooltip='Bounding box'
).add_to(m)

fig.add_child(m)
fig

### 💡 Example 11.2 — HeatMap Plugin

In [ ]:
import folium
import numpy as np
from folium import Figure
from folium.plugins import HeatMap

# Simulate 200 random points within North Dakota
rng = np.random.default_rng(42)
lat_points = rng.uniform(45.9, 49.0, 200)
lon_points = rng.uniform(-104.1, -96.6, 200)
weights    = rng.uniform(0.2, 1.0, 200)

heat_data = list(zip(lat_points, lon_points, weights))

fig = Figure(width=800, height=500)
m   = folium.Map(location=[47.5, -100.5], zoom_start=7, tiles='CartoDB dark_matter')

HeatMap(
    heat_data,
    radius=15,
    blur=10,
    max_zoom=10
).add_to(m)

fig.add_child(m)
fig

### 💡 Example 11.3 — MarkerCluster for Dense Point Data

In [ ]:
import folium
import numpy as np
from folium import Figure
from folium.plugins import MarkerCluster

rng = np.random.default_rng(7)
lats = rng.uniform(45.9, 49.0, 100)
lons = rng.uniform(-104.1, -96.6, 100)

fig = Figure(width=800, height=500)
m   = folium.Map(location=[47.5, -100.5], zoom_start=7)

cluster = MarkerCluster(name='Sample Points').add_to(m)

for i, (lat, lon) in enumerate(zip(lats, lons)):
    folium.Marker(
        [lat, lon],
        popup=f'Point {i+1}',
        tooltip=f'({lat:.2f}, {lon:.2f})'
    ).add_to(cluster)

folium.LayerControl().add_to(m)
fig.add_child(m)
fig

---

# 🎉 Congratulations — All 12 Sections Complete!

---

| ✅ | Section |
|----|----------|
| ✅ | 0 — Setup & Environment |
| ✅ | 1 — Basic Map Initialization |
| ✅ | 2 — Markers & Popups |
| ✅ | 3 — Customizing Markers with Icons |
| ✅ | 4 — Routing with OpenRouteService API |
| ✅ | 5 — Parsing GeoJSON & Plotting Routes |
| ✅ | 6 — Saving Maps as HTML |
| ✅ | 7 — Downloading Data with pygris |
| ✅ | 8 — Multi-Layer Maps with .explore() |
| ✅ | 9 — Road Network Categorization & Styling |
| ✅ | 10 — Layer Controls & Basemap Switching |
| ✅ | 11 — Bonus: Circles, Polygons & Heatmaps |

---

## 🚀 Next Steps

| Topic | Libraries |
|-------|-----------|
| Advanced interactive maps | `leafmap`, `kepler.gl`, `pydeck` |
| Routing & isochrones | `openrouteservice`, `osmnx` |
| Satellite basemaps | `contextily`, `xyzservices` |
| 3D web maps | `pydeck`, `deck.gl` |
| Dashboard embedding | `streamlit`, `dash`, `panel` |

---

### Happy Mapping! 🌐💻

*Every marker tells a story — Folium helps you share it interactively.*

---